In [3]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/chrisyoko/semrel2024-raw/raw/dataset_summary.json
/kaggle/input/datasets/chrisyoko/semrel2024-raw/raw/afr/dev.csv
/kaggle/input/datasets/chrisyoko/semrel2024-raw/raw/afr/test.csv
/kaggle/input/datasets/chrisyoko/semrel2024-raw/raw/hau/dev.csv
/kaggle/input/datasets/chrisyoko/semrel2024-raw/raw/hau/train.csv
/kaggle/input/datasets/chrisyoko/semrel2024-raw/raw/hau/test.csv
/kaggle/input/datasets/chrisyoko/semrel2024-raw/raw/eng/dev.csv
/kaggle/input/datasets/chrisyoko/semrel2024-raw/raw/eng/train.csv
/kaggle/input/datasets/chrisyoko/semrel2024-raw/raw/eng/test.csv
/kaggle/input/datasets/chrisyoko/semrel2024-raw/raw/kin/dev.csv
/kaggle/input/datasets/chrisyoko/semrel2024-raw/raw/kin/train.csv
/kaggle/input/datasets/chrisyoko/semrel2024-raw/raw/kin/test.csv


In [4]:
# Verify GPU
import torch
print(f"GPU available: {torch.cuda.is_available()}")
print(f"GPU name: {torch.cuda.get_device_name(0)}")

# Install any missing packages
import subprocess
subprocess.run(['pip', 'install', 'sentence-transformers', '-q'])

GPU available: True
GPU name: Tesla T4


CompletedProcess(args=['pip', 'install', 'sentence-transformers', '-q'], returncode=0)

In [5]:
import pandas as pd
import numpy as np
import torch
import csv
import os
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
from transformers import get_linear_schedule_with_warmup
from scipy.stats import spearmanr
from sklearn.metrics import mean_squared_error

# ── Config ────────────────────────────────────────────────────────────
DATA_ROOT = '/kaggle/input/datasets/chrisyoko/semrel2024-raw/raw'
RESULTS_LOG = '/kaggle/working/results_log.csv'
CKPT_DIR    = '/kaggle/working/checkpoints'
MODEL_NAME  = 'xlm-roberta-base'
MAX_LEN     = 128
BATCH_SIZE  = 16
EPOCHS      = 10
LR          = 2e-5
PATIENCE    = 3
DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

os.makedirs(CKPT_DIR, exist_ok=True)
print(f"Device: {DEVICE}")
print(f"Model:  {MODEL_NAME}")

Device: cuda
Model:  xlm-roberta-base


In [6]:
class SemRelDataset(Dataset):
    def __init__(self, df, tokenizer, max_len):
        self.df        = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        enc = self.tokenizer(
            str(row['sentence1']),
            str(row['sentence2']),
            truncation=True,
            max_length=self.max_len,
            padding='max_length',
            return_tensors='pt'
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(),
            'attention_mask': enc['attention_mask'].squeeze(),
            'label':          torch.tensor(row['label'], dtype=torch.float)
        }

print("SemRelDataset class defined.")

SemRelDataset class defined.


In [7]:
from transformers import XLMRobertaModel

class SemRelModel(torch.nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.encoder = XLMRobertaModel.from_pretrained(model_name)
        self.regressor = torch.nn.Linear(self.encoder.config.hidden_size, 1)
        self.dropout = torch.nn.Dropout(0.1)

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls = outputs.last_hidden_state[:, 0, :]  # [CLS] token
        return self.regressor(self.dropout(cls)).squeeze(-1)


def evaluate(model, loader):
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for batch in loader:
            ids  = batch['input_ids'].to(DEVICE)
            mask = batch['attention_mask'].to(DEVICE)
            out  = model(ids, mask).cpu().numpy()
            preds.extend(out)
            labels.extend(batch['label'].numpy())
    rho, _ = spearmanr(preds, labels)
    mse    = mean_squared_error(labels, preds)
    return rho, mse, preds


def train(lang_code, lang_name, tokenizer):
    print(f"\n{'='*60}")
    print(f"  Training XLM-R on {lang_name}")
    print(f"{'='*60}")

    train_df = pd.read_csv(f'{DATA_ROOT}/{lang_code}/train.csv')
    dev_df   = pd.read_csv(f'{DATA_ROOT}/{lang_code}/dev.csv')
    test_df  = pd.read_csv(f'{DATA_ROOT}/{lang_code}/test.csv')

    train_loader = DataLoader(SemRelDataset(train_df, tokenizer, MAX_LEN),
                              batch_size=BATCH_SIZE, shuffle=True)
    dev_loader   = DataLoader(SemRelDataset(dev_df,   tokenizer, MAX_LEN),
                              batch_size=BATCH_SIZE)
    test_loader  = DataLoader(SemRelDataset(test_df,  tokenizer, MAX_LEN),
                              batch_size=BATCH_SIZE)

    model     = SemRelModel(MODEL_NAME).to(DEVICE)
    optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    total_steps = len(train_loader) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(0.1 * total_steps),
        num_training_steps=total_steps
    )
    loss_fn = torch.nn.MSELoss()

    best_rho, patience_counter, best_epoch = -1, 0, 0

    for epoch in range(1, EPOCHS + 1):
        model.train()
        total_loss = 0
        for batch in train_loader:
            ids   = batch['input_ids'].to(DEVICE)
            mask  = batch['attention_mask'].to(DEVICE)
            lbls  = batch['label'].to(DEVICE)
            optimizer.zero_grad()
            preds = model(ids, mask)
            loss  = loss_fn(preds, lbls)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            total_loss += loss.item()

        dev_rho, dev_mse, _ = evaluate(model, dev_loader)
        avg_loss = total_loss / len(train_loader)
        print(f"  Epoch {epoch:2d} | loss={avg_loss:.4f} | dev ρ={dev_rho:.4f} | dev MSE={dev_mse:.4f}")

        if dev_rho > best_rho:
            best_rho = dev_rho
            best_epoch = epoch
            patience_counter = 0
            torch.save(model.state_dict(),
                       f'{CKPT_DIR}/xlmr_{lang_code}_best.pt')
            print(f"           ✓ New best saved (ρ={best_rho:.4f})")
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                print(f"  Early stopping at epoch {epoch} (patience={PATIENCE})")
                break

    # Evaluate best checkpoint on test set
    model.load_state_dict(torch.load(f'{CKPT_DIR}/xlmr_{lang_code}_best.pt'))
    test_rho, test_mse, _ = evaluate(model, test_loader)
    print(f"\n  Best epoch: {best_epoch} | Test ρ={test_rho:.4f} | Test MSE={test_mse:.4f}")
    return test_rho, test_mse


print("Training functions defined.")

Training functions defined.


In [8]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Train on English first — validate the pipeline
rho, mse = train('eng', 'English', tokenizer)

# Log result
with open(RESULTS_LOG, 'a', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=[
        'experiment_id','model','model_variant','language',
        'split','spearman_rho','mse','augmented','notes'
    ])
    writer.writerow({
        'experiment_id': 'TL-ENG',
        'model': 'xlmr_finetuned',
        'model_variant': 'xlm-roberta-base',
        'language': 'English',
        'split': 'test',
        'spearman_rho': round(rho, 4),
        'mse': round(mse, 4),
        'augmented': False,
        'notes': 'Fine-tuned on English train set. Best checkpoint on dev Spearman.'
    })

print(f"\n✓ English XLM-R complete: ρ={rho:.4f}, MSE={mse:.4f}")
print("✓ Logged to results_log.csv")

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]


  Training XLM-R on English


model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Epoch  1 | loss=0.1659 | dev ρ=0.7178 | dev MSE=0.0386
           ✓ New best saved (ρ=0.7178)
  Epoch  2 | loss=0.0352 | dev ρ=0.7919 | dev MSE=0.0250
           ✓ New best saved (ρ=0.7919)
  Epoch  3 | loss=0.0242 | dev ρ=0.7913 | dev MSE=0.0222
  Epoch  4 | loss=0.0163 | dev ρ=0.7960 | dev MSE=0.0200
           ✓ New best saved (ρ=0.7960)
  Epoch  5 | loss=0.0130 | dev ρ=0.8104 | dev MSE=0.0211
           ✓ New best saved (ρ=0.8104)
  Epoch  6 | loss=0.0100 | dev ρ=0.8078 | dev MSE=0.0219
  Epoch  7 | loss=0.0084 | dev ρ=0.8011 | dev MSE=0.0257
  Epoch  8 | loss=0.0068 | dev ρ=0.8028 | dev MSE=0.0318
  Early stopping at epoch 8 (patience=3)

  Best epoch: 5 | Test ρ=0.8353 | Test MSE=0.0200

✓ English XLM-R complete: ρ=0.8353, MSE=0.0200
✓ Logged to results_log.csv


In [9]:
# Train on Hausa and Kinyarwanda
for lang_code, lang_name in [('hau', 'Hausa'), ('kin', 'Kinyarwanda')]:
    rho, mse = train(lang_code, lang_name, tokenizer)

    with open(RESULTS_LOG, 'a', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=[
            'experiment_id','model','model_variant','language',
            'split','spearman_rho','mse','augmented','notes'
        ])
        writer.writerow({
            'experiment_id': f'TL-2-{lang_code}',
            'model': 'xlmr_finetuned',
            'model_variant': 'xlm-roberta-base',
            'language': lang_name,
            'split': 'test',
            'spearman_rho': round(rho, 4),
            'mse': round(mse, 4),
            'augmented': False,
            'notes': f'Fine-tuned on {lang_name} train set. Best checkpoint on dev Spearman.'
        })
    print(f"✓ {lang_name} logged: ρ={rho:.4f}, MSE={mse:.4f}")


  Training XLM-R on Hausa


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Epoch  1 | loss=0.0973 | dev ρ=0.6072 | dev MSE=0.0586
           ✓ New best saved (ρ=0.6072)
  Epoch  2 | loss=0.0885 | dev ρ=0.6861 | dev MSE=0.0424
           ✓ New best saved (ρ=0.6861)
  Epoch  3 | loss=0.0653 | dev ρ=0.6769 | dev MSE=0.0512
  Epoch  4 | loss=0.0626 | dev ρ=0.7565 | dev MSE=0.0377
           ✓ New best saved (ρ=0.7565)
  Epoch  5 | loss=0.0549 | dev ρ=0.7565 | dev MSE=0.0343
           ✓ New best saved (ρ=0.7565)
  Epoch  6 | loss=0.0474 | dev ρ=0.7430 | dev MSE=0.0334
  Epoch  7 | loss=0.0464 | dev ρ=0.7693 | dev MSE=0.0321
           ✓ New best saved (ρ=0.7693)
  Epoch  8 | loss=0.0402 | dev ρ=0.7658 | dev MSE=0.0323
  Epoch  9 | loss=0.0371 | dev ρ=0.7734 | dev MSE=0.0305
           ✓ New best saved (ρ=0.7734)
  Epoch 10 | loss=0.0358 | dev ρ=0.7763 | dev MSE=0.0306
           ✓ New best saved (ρ=0.7763)

  Best epoch: 10 | Test ρ=0.6862 | Test MSE=0.0404
✓ Hausa logged: ρ=0.6862, MSE=0.0404

  Training XLM-R on Kinyarwanda


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Epoch  1 | loss=0.0575 | dev ρ=0.0797 | dev MSE=0.0751
           ✓ New best saved (ρ=0.0797)
  Epoch  2 | loss=0.0562 | dev ρ=0.2066 | dev MSE=0.0702
           ✓ New best saved (ρ=0.2066)
  Epoch  3 | loss=0.0564 | dev ρ=0.2786 | dev MSE=0.0657
           ✓ New best saved (ρ=0.2786)
  Epoch  4 | loss=0.0498 | dev ρ=0.4566 | dev MSE=0.0598
           ✓ New best saved (ρ=0.4566)
  Epoch  5 | loss=0.0500 | dev ρ=0.5581 | dev MSE=0.0551
           ✓ New best saved (ρ=0.5581)
  Epoch  6 | loss=0.0427 | dev ρ=0.5839 | dev MSE=0.0579
           ✓ New best saved (ρ=0.5839)
  Epoch  7 | loss=0.0429 | dev ρ=0.5951 | dev MSE=0.0505
           ✓ New best saved (ρ=0.5951)
  Epoch  8 | loss=0.0408 | dev ρ=0.6407 | dev MSE=0.0471
           ✓ New best saved (ρ=0.6407)
  Epoch  9 | loss=0.0367 | dev ρ=0.6440 | dev MSE=0.0468
           ✓ New best saved (ρ=0.6440)
  Epoch 10 | loss=0.0360 | dev ρ=0.6329 | dev MSE=0.0484

  Best epoch: 9 | Test ρ=0.6339 | Test MSE=0.0269
✓ Kinyarwanda logged: ρ=0.63

In [10]:
from sentence_transformers import SentenceTransformer
# Zero-shot: load English-fine-tuned XLM-R, evaluate on African language test sets

def evaluate_zero_shot(lang_code, lang_name):
    print(f"\nZero-shot eval on {lang_name}...")
    test_df  = pd.read_csv(f'{DATA_ROOT}/{lang_code}/test.csv')
    test_loader = DataLoader(
        SemRelDataset(test_df, tokenizer, MAX_LEN),
        batch_size=BATCH_SIZE
    )
    # Load English checkpoint
    model = SemRelModel(MODEL_NAME).to(DEVICE)
    model.load_state_dict(torch.load(f'{CKPT_DIR}/xlmr_eng_best.pt'))
    rho, mse, _ = evaluate(model, test_loader)
    print(f"  {lang_name}: ρ={rho:.4f}, MSE={mse:.4f}")
    return rho, mse

for lang_code, lang_name in [('afr','Afrikaans'), ('hau','Hausa'), ('kin','Kinyarwanda')]:
    rho, mse = evaluate_zero_shot(lang_code, lang_name)
    with open(RESULTS_LOG, 'a', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=[
            'experiment_id','model','model_variant','language',
            'split','spearman_rho','mse','augmented','notes'
        ])
        writer.writerow({
            'experiment_id': f'TL-1-{lang_code}',
            'model': 'xlmr_zeroshot',
            'model_variant': 'xlm-roberta-base',
            'language': lang_name,
            'split': 'test',
            'spearman_rho': round(rho, 4),
            'mse': round(mse, 4),
            'augmented': False,
            'notes': 'Zero-shot: English fine-tuned checkpoint evaluated on African language.'
        })


Zero-shot eval on Afrikaans...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Afrikaans: ρ=0.8177, MSE=0.0159

Zero-shot eval on Hausa...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Hausa: ρ=0.6422, MSE=0.0539

Zero-shot eval on Kinyarwanda...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Kinyarwanda: ρ=0.4819, MSE=0.0349


In [11]:
# Retrain Kinyarwanda with more epochs
EPOCHS = 15  # temporarily increase
rho, mse = train('kin', 'Kinyarwanda', tokenizer)
EPOCHS = 10  # reset

with open(RESULTS_LOG, 'a', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=[
        'experiment_id','model','model_variant','language',
        'split','spearman_rho','mse','augmented','notes'
    ])
    writer.writerow({
        'experiment_id': 'TL-2-kin-v2',
        'model': 'xlmr_finetuned',
        'model_variant': 'xlm-roberta-base',
        'language': 'Kinyarwanda',
        'split': 'test',
        'spearman_rho': round(rho, 4),
        'mse': round(mse, 4),
        'augmented': False,
        'notes': 'Fine-tuned Kinyarwanda with 15 epochs — model still converging at epoch 10.'
    })
print(f"✓ Kinyarwanda v2: ρ={rho:.4f}, MSE={mse:.4f}")


  Training XLM-R on Kinyarwanda


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Epoch  1 | loss=0.2620 | dev ρ=0.0552 | dev MSE=0.0682
           ✓ New best saved (ρ=0.0552)
  Epoch  2 | loss=0.0688 | dev ρ=0.1801 | dev MSE=0.0668
           ✓ New best saved (ρ=0.1801)
  Epoch  3 | loss=0.0581 | dev ρ=0.2555 | dev MSE=0.0667
           ✓ New best saved (ρ=0.2555)
  Epoch  4 | loss=0.0543 | dev ρ=0.2862 | dev MSE=0.0656
           ✓ New best saved (ρ=0.2862)
  Epoch  5 | loss=0.0506 | dev ρ=0.3387 | dev MSE=0.0607
           ✓ New best saved (ρ=0.3387)
  Epoch  6 | loss=0.0505 | dev ρ=0.4392 | dev MSE=0.0598
           ✓ New best saved (ρ=0.4392)
  Epoch  7 | loss=0.0462 | dev ρ=0.5065 | dev MSE=0.1109
           ✓ New best saved (ρ=0.5065)
  Epoch  8 | loss=0.0381 | dev ρ=0.6202 | dev MSE=0.0606
           ✓ New best saved (ρ=0.6202)
  Epoch  9 | loss=0.0327 | dev ρ=0.6203 | dev MSE=0.0468
           ✓ New best saved (ρ=0.6203)
  Epoch 10 | loss=0.0295 | dev ρ=0.5654 | dev MSE=0.0517
  Epoch 11 | loss=0.0300 | dev ρ=0.6115 | dev MSE=0.0522
  Epoch 12 | loss=0.02

In [12]:
import os
for f in os.listdir('/kaggle/working'):
    print(f)

.virtual_documents
checkpoints
results_log.csv


In [ ]:
# ── Always run this last cell before ending session ──
import pandas as pd

df = pd.read_csv('/kaggle/working/results_log.csv')
print(f"\n✓ {len(df)} results saved to results_log.csv")
print(df[['experiment_id','language','spearman_rho','mse']].to_string(index=False))

# This file will be available in Output panel for download
print("\n✓ Download results_log.csv from the Output panel on the right.")